# RARE26 — cross-center fine-tune (GPU)
The frozen-probe ceiling is ~0.28 LOCO-mean PPV@90R. This fine-tunes the last K DINOv2 blocks with heavy augmentation + a 90R tail loss to learn center-invariant features.

**Steps:** Runtime → Change runtime type → **GPU (A100/L4/T4)**. Upload `rare26_finetune.tar.gz` (from `bash phase3/pack_for_cloud.sh`) to your Drive, then run the cells.

In [ ]:
!nvidia-smi -L
!pip -q install timm==1.0.27

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
# adjust the path to where you uploaded the tarball:
!cp '/content/drive/MyDrive/rare26_finetune.tar.gz' /content/ && tar -xzf /content/rare26_finetune.tar.gz -C /content/
%cd /content/rare26
!ls phase3 && python -c "import torch;print('cuda',torch.cuda.is_available())"

## LOCO legs — estimate the new-center gain over the 0.28 frozen ceiling

In [ ]:
# hold out center_2 (train on center_1). Add --neg-list dataset/unl_neg.txt if you packed with-negs.
!python -m phase3.finetune --holdout center_2 --unfreeze 4 --epochs 40 --bs 64 --lr 1e-4 \
    --loss bce+rank+pauc --warmup 3 --out phase3/cache/ft_holdout_c2.pt

In [ ]:
# hold out center_1 (the hard leg). Watch the printed LOCO-val PPV@90R per epoch.
!python -m phase3.finetune --holdout center_1 --unfreeze 4 --epochs 40 --bs 64 --lr 1e-4 \
    --loss bce+rank+pauc --warmup 3 --out phase3/cache/ft_holdout_c1.pt

## Ship — train on BOTH centers (more domain diversity → best for an unseen center)

In [ ]:
!python -m phase3.finetune --holdout none --unfreeze 4 --epochs 35 --bs 64 --lr 1e-4 \
    --loss bce+rank+pauc --warmup 3 --out phase3/cache/ft_ship.pt
from google.colab import files; files.download('phase3/cache/ft_ship.pt')

**Report back** the per-epoch `LOCO-val PPV@90R` for both legs. If fine-tune beats the frozen 0.28 LOCO-mean, we tune further (more unfrozen blocks, EMA, pAUC weight, stronger aug). If it overfits (LOCO-val drops while train loss falls), reduce `--unfreeze` to 2, raise `--wd`, or add `--neg-list`.